# Global predictability ceilings for lag-based emulation of NASA POWER soil surface wetness

Global panel: 250 POWER cells selected, 249 analysed after one permanently saturated cell is dropped in Cell 3. The linear baseline vs. the ceiling is the paper's spine.

**Inputs:** 24 GEE export chunks `gpc_0.csv` … `gpc_23.csv` (6 longitude sectors × 4 latitude bands) in `data/`

**Every heavy cell checkpoints per grid cell.** If Colab recycles, restore the zip and rerun the
same cell — completed cells are skipped.

Run order: 0 → 1 → 2 → **Z** → 3 → 4 → 5 → **Z** → 6 → 7 → 7b → **Z** → 8 → 9 → 10
(**Z** = the checkpoint cell; run it often.)


## Cell 0 — setup

In [ ]:
USE_DRIVE = False      # True to persist across runtime restarts

from pathlib import Path
import sys
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    assert Path("/content/drive/MyDrive").exists(), "Drive did not mount"
    %cd /content/drive/MyDrive
    !mkdir -p ssw_global/data
    %cd ssw_global

Path("data").mkdir(exist_ok=True)
NEED = [f"gpc_{i}.csv" for i in range(24)]
missing = [f for f in NEED if not (Path("data")/f).exists()]

if missing and IN_COLAB:
    hits = list(Path("/content/drive/MyDrive").rglob("gpc_*.csv")) \
           if Path("/content/drive/MyDrive").exists() else []
    for h in hits:
        if h.name in missing: (Path("data")/h.name).write_bytes(h.read_bytes())
    missing = [f for f in NEED if not (Path("data")/f).exists()]
    if missing:
        print(f"Upload the {len(missing)} chunk CSVs:", missing[:5], "...")
        from google.colab import files
        up = files.upload()
        for k in up: (Path("data")/k).write_bytes(up[k])

!pip install -q pyarrow lightgbm xgboost statsmodels

print("\n--- SETUP ---")
ok = True
for f in NEED:
    p = Path("data")/f
    ok &= p.exists()
print(f"  {sum((Path('data')/f).exists() for f in NEED)}/24 chunk CSVs present")
assert ok, "Missing chunk CSVs — rerun and upload all 24"
print("ready")

## Cell Z — checkpoint (run often)

Zips `data/` and downloads it. Restore with the cell below it.

In [ ]:
import shutil
from pathlib import Path
shutil.make_archive("ssw_global_results", "zip", "data")
try:
    from google.colab import files; files.download("ssw_global_results.zip")
    print("downloaded ssw_global_results.zip")
except ImportError:
    print("saved", Path("ssw_global_results.zip").resolve())

# --- TO RESTORE after a recycle, run this instead: ---
# from google.colab import files; import shutil
# from pathlib import Path; Path("data").mkdir(exist_ok=True)
# up = files.upload(); shutil.unpack_archive("ssw_global_results.zip", "data")

## Cell 1 — global panel

Tested against the real 24-chunk export (50,860 rows). Handles three things the raw export
needs: chunk-boundary duplicates, object dtype from near-empty chunks, and PET_ann = 0 in cold
cells (which sends the aridity index to ~15,000).

Replication cells (Punjab) are forced in first and exempt from the separation rule — Moga and
Firozpur are 59.6 km apart, just inside one grid step.

**Verified on your data:** 250 cells, 29 strata, corr(AI, irrig) = -0.075, min separation 55.1 km.


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
OUT = Path("data")
TARGET, MIN_PER_STRATUM, T_MIN, MIN_SEP_KM, R = 250, 4, 5.0, 55.0, 6371.0

g = pd.concat([pd.read_csv(f) for f in sorted(OUT.glob("gpc_*.csv"))], ignore_index=True)
n0 = len(g)
NUM = ["lat","lon","land_frac","crop_frac","AI","P_ann","PET_ann","T2M_ann","ndvi_dry_crop"]
for c in NUM: g[c] = pd.to_numeric(g[c], errors="coerce")   # near-empty chunks -> object dtype
g = g.dropna(subset=["lat","lon","AI","T2M_ann","PET_ann"])
g = g.drop_duplicates(subset=["lat","lon"])                  # chunk boundaries overlap
n1 = len(g)
# PET_ann == 0 in cold cells sends AI to ~15000; T and PET filters both required
g = g[(g.land_frac>0.2) & (g.T2M_ann>=T_MIN) & (g.PET_ann>200) & (g.P_ann>0)].reset_index(drop=True)

g["crop_frac"]   = g.crop_frac.fillna(0.0)
g["irrig_index"] = np.where(g.crop_frac<0.02, 0.0, g.ndvi_dry_crop.fillna(0.0)*g.crop_frac)
cropped = g.irrig_index > 0
q1, q2 = g.loc[cropped,"irrig_index"].quantile([1/3, 2/3])
g["irrig_class"] = np.select([~cropped, g.irrig_index<=q1, g.irrig_index<=q2],
                             ["low","low","med"], default="high")
g["aridity_class"] = pd.cut(g.AI, [-np.inf,0.05,0.20,0.50,0.65,np.inf],
                            labels=["hyperarid","arid","semiarid","dry-subhumid","humid"])
g["hemi"] = np.where(g.lat>=0, "N", "S")
g["stratum"] = g.aridity_class.astype(str) + "|" + g.irrig_class + "|" + g.hemi

PUNJAB = {"Moga":(30.816,75.171),"Hoshiarpur":(31.514,75.911),"Firozpur":(30.923,74.610)}
g["name"], g["replication"] = None, False
for nm,(la,lo) in PUNJAB.items():
    i = np.hypot(g.lat-la, g.lon-lo).idxmin(); g.loc[i,"name"]=nm; g.loc[i,"replication"]=True

def xyz(df):
    la, lo = np.radians(df.lat.to_numpy()), np.radians(df.lon.to_numpy())
    return np.column_stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)])

def fp(df, k, seed):
    P = xyz(df); ch = list(seed) or [0]
    while len(ch) < min(k, len(df)):
        d = np.min(np.linalg.norm(P[:,None,:]-P[ch][None,:,:], axis=2), axis=1); d[ch] = -1
        ch.append(int(np.argmax(d)))
    return ch

counts = g.stratum.value_counts()
alloc = {s: min(MIN_PER_STRATUM, n) for s,n in counts.items()}
left = TARGET - sum(alloc.values()); CAP = max(MIN_PER_STRATUM+2, TARGET//len(counts)+2)
while left > 0:
    room = [s for s in counts.index if alloc[s] < min(counts[s], CAP)]
    if not room: CAP += 2; continue
    for s in room:
        if left <= 0: break
        alloc[s] += 1; left -= 1

kept  = list(g.index[g.replication])          # replication cells forced in, exempt from spacing
keptP = list(xyz(g.loc[kept])) if kept else []
def far(i):
    if not keptP: return True
    return (np.linalg.norm(np.array(keptP)-xyz(g.loc[[i]])[0], axis=1)*R).min() >= MIN_SEP_KM

for s_, k in alloc.items():
    sub = g[g.stratum==s_]
    if len(sub) == 0: continue
    have = sum(1 for i in kept if g.at[i,"stratum"]==s_)
    order = fp(sub.reset_index(), min(len(sub), k*4), [])
    for i in sub.reset_index().loc[order,"index"].tolist():
        if have >= k: break
        if i in kept: continue
        if far(i): kept.append(i); keptP.append(xyz(g.loc[[i]])[0]); have += 1

panel = g.loc[sorted(set(kept))].copy().reset_index(drop=True)
panel["cell_id"] = [f"{a:+07.3f}_{b:+08.3f}".replace(".","p") for a,b in zip(panel.lat, panel.lon)]
panel.to_parquet(OUT/"panel_global.parquet", index=False)

print("--- SANITY ---")
print(f"raw {n0} -> dedup {n1} -> land+warm+validPET {len(g)} -> panel {len(panel)} (target {TARGET})")
print("dupes:", panel.duplicated(["lat","lon"]).sum(), "| Punjab:", int(panel.replication.sum()), "(want 3)")
print("strata: %d of %d" % (panel.stratum.nunique(), g.stratum.nunique()))
print("\naridity x irrigation:\n", pd.crosstab(panel.aridity_class, panel.irrig_class).to_string())
print("\nhemisphere:", panel.hemi.value_counts().to_dict())
print("AI %.3f-%.3f | P %.0f-%.0f mm | PET %.0f-%.0f | T %.1f-%.1f C"
      % (panel.AI.min(),panel.AI.max(),panel.P_ann.min(),panel.P_ann.max(),
         panel.PET_ann.min(),panel.PET_ann.max(),panel.T2M_ann.min(),panel.T2M_ann.max()))
r_,p_ = stats.pearsonr(panel.AI, panel.irrig_index)
rl,pl = stats.pearsonr(np.log(panel.AI), panel.irrig_index)
print("\n*** corr(AI, irrig_index)    = %.3f (p=%.2g)" % (r_,p_))
print("*** corr(logAI, irrig_index) = %.3f (p=%.2g)   <- want |r| < 0.6" % (rl,pl))
Pm = xyz(panel); D = np.linalg.norm(Pm[:,None,:]-Pm[None,:,:], axis=2); np.fill_diagonal(D, np.inf)
print("min separation: %.1f km" % (D.min()*R))
print("\nPunjab:\n", panel[panel.replication][["name","lat","lon","AI","aridity_class","irrig_class"]].round(3).to_string(index=False))

## Cell 2 — POWER daily pull

~250 cells, checkpointed to `data/raw_daily/`. Budget 10–20 min. **Run Cell Z after this.**

In [ ]:
import requests, time
RAW = OUT/"raw_daily"; RAW.mkdir(parents=True, exist_ok=True)
panel = pd.read_parquet(OUT/"panel_global.parquet").reset_index(drop=True)

PARAMS = ["GWETTOP","GWETROOT","GWETPROF","PRECTOTCORR","T2M","T2M_MAX","T2M_MIN",
          "RH2M","WS2M","ALLSKY_SFC_SW_DWN","EVLAND","PS"]
URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
START, END = "19810101", "20251231"

done, failed = 0, []
for i, r in panel.iterrows():
    fp_ = RAW/f"{r.cell_id}.parquet"
    if fp_.exists(): done += 1; continue
    try:
        q = dict(parameters=",".join(PARAMS), community="AG", latitude=float(r.lat),
                 longitude=float(r.lon), start=START, end=END, format="JSON")
        resp = requests.get(URL, params=q, timeout=300); resp.raise_for_status()
        df = pd.DataFrame(resp.json()["properties"]["parameter"])
        df.index = pd.to_datetime(df.index, format="%Y%m%d")
        df = df.replace(-999.0, np.nan).rename_axis("date").reset_index()
        df.insert(0, "cell_id", r.cell_id)
        df.insert(1, "name", r["name"] if pd.notna(r["name"]) else r.cell_id)
        df.to_parquet(fp_, index=False); done += 1; time.sleep(1.2)
    except Exception as e:
        failed.append((r.cell_id, repr(e)[:80]))
    if i % 25 == 0: print(f"  {i}/{len(panel)} ok={done} fail={len(failed)}")

daily = pd.concat([pd.read_parquet(f) for f in sorted(RAW.glob("*.parquet"))], ignore_index=True)
daily.to_parquet(OUT/"power_daily_panel.parquet", index=False)

print("\n--- SANITY ---")
print("cells:", daily.cell_id.nunique(), "of", len(panel), "| failures:", len(failed))
if failed: print("  first:", failed[:3])
n = daily.groupby("cell_id").size()
print("rows/cell: min %d max %d (must be equal)" % (n.min(), n.max()))
print("GWETTOP: %.3f - %.3f in [0,1]? %s"
      % (daily.GWETTOP.min(), daily.GWETTOP.max(), bool(daily.GWETTOP.between(0,1).all())))
gaps = {c:int(gg.date.diff().dt.days.fillna(1).ne(1).sum()) for c,gg in daily.groupby("cell_id")}
print("cells with date gaps:", sum(v>0 for v in gaps.values()), "(must be 0)")
j = panel.set_index("cell_id").join(daily.groupby("cell_id").GWETTOP.mean().rename("gw"))
print("\nGWETTOP by aridity (must INCREASE):")
print(j.groupby("aridity_class", observed=True).gw.agg(["count","mean"]).round(3).to_string())
print("*** corr(AI, GWETTOP) = %.3f  (want clearly positive)" % j[["AI","gw"]].corr().iloc[0,1])

## Cell 3 — monthly panel + Stage A diagnostics

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

daily = pd.read_parquet(OUT/"power_daily_panel.parquet")
panel = pd.read_parquet(OUT/"panel_global.parquet")
daily["date"] = pd.to_datetime(daily.date)

AGG = {"GWETTOP":"mean","GWETROOT":"mean","GWETPROF":"mean","PRECTOTCORR":"sum",
       "T2M":"mean","T2M_MAX":"mean","T2M_MIN":"mean","RH2M":"mean","WS2M":"mean",
       "ALLSKY_SFC_SW_DWN":"mean","EVLAND":"mean","PS":"mean"}
mon = (daily.set_index("date").groupby(["cell_id","name"]).resample("MS").agg(AGG)
             .reset_index().rename(columns={"date":"month"}))
mon = mon[mon.month >= "1984-01-01"].reset_index(drop=True)

# Degenerate-cell drop (paper Sec 2.2), as run: GWETTOP is censored at 1.0, and one cell
# (7.5S 34.7W) returns exactly 1.0 in all 504 months. A cell with no variance carries no
# predictable signal and gives undefined ACF and sample entropy. Drop cells with sd < 0.01
# from the monthly series AND the panel file, so every downstream stage sees 249 cells.
MIN_SD = 0.01
sd_ = mon.groupby("cell_id").GWETTOP.std()
DEGENERATE = sd_[sd_ < MIN_SD].index.tolist()
print("dropping", len(DEGENERATE), "degenerate cell(s):", DEGENERATE)
mon   = mon[~mon.cell_id.isin(DEGENERATE)].reset_index(drop=True)
panel = panel[~panel.cell_id.isin(DEGENERATE)].reset_index(drop=True)
panel.to_parquet(OUT/"panel_global.parquet", index=False)
mon.to_parquet(OUT/"power_monthly_panel.parquet", index=False)

def harmonics(t, n_h=2, period=12):
    return np.column_stack([f(2*np.pi*k*t/period) for k in range(1,n_h+1) for f in (np.sin,np.cos)])

def sampen(x, m=2, r=0.2):
    x = (x-x.mean())/x.std(); N = len(x)
    def phi(mm):
        X = np.array([x[i:i+mm] for i in range(N-mm+1)])
        D = np.max(np.abs(X[:,None,:]-X[None,:,:]), axis=2); np.fill_diagonal(D, np.inf)
        return (D <= r).sum()
    A, B = phi(m+1), phi(m)
    return np.inf if A == 0 else -np.log(A/B)

rows = []
for cid, gg in mon.groupby("cell_id"):
    gg = gg.sort_values("month"); y = gg.GWETTOP.to_numpy(float); t = np.arange(len(y))
    H = np.column_stack([np.ones(len(y)), harmonics(t)])
    b, *_ = np.linalg.lstsq(H, y, rcond=None); resid = y - H@b
    a_raw, a_des = acf(y, nlags=24, fft=True), acf(resid, nlags=24, fft=True)
    p_des = pacf(resid, nlags=12, method="ywadjusted")
    rows.append(dict(cell_id=cid, n_months=len(y), mean=y.mean(), sd=y.std(),
        seas_share=1-resid.var()/y.var(), acf1_raw=a_raw[1], acf12_raw=a_raw[12],
        acf1_des=a_des[1], acf2_des=a_des[2], acf12_des=a_des[12],
        pacf1_des=p_des[1], sampen_des=sampen(resid), resid_sd=resid.std()))

diag = pd.DataFrame(rows).merge(panel[["cell_id","name","lat","lon","AI","aridity_class",
    "irrig_index","irrig_class","hemi","replication"]], on="cell_id")
diag.to_parquet(OUT/"stageA_diagnostics.parquet", index=False)

print("--- SANITY ---")
print("cells analysed:", mon.cell_id.nunique(), "(want 249)")
print("monthly rows:", len(mon), "| months/cell:", mon.groupby("cell_id").size().unique())
print("GWETTOP in [0,1]?", bool(mon.GWETTOP.between(0,1).all()))
print("seas_share %.3f-%.3f | acf1_des %.3f-%.3f"
      % (diag.seas_share.min(),diag.seas_share.max(),diag.acf1_des.min(),diag.acf1_des.max()))
print("\nby aridity:\n", diag.groupby("aridity_class",observed=True)[["seas_share","acf1_des","sampen_des"]].mean().round(3).to_string())
print("\nby hemisphere:\n", diag.groupby("hemi")[["seas_share","acf1_des"]].mean().round(3).to_string())
print("\nPunjab:\n", diag[diag.replication][["name","seas_share","acf1_des","sampen_des"]].round(3).to_string(index=False))

## Cell 4 — Stage B: predictability ceilings

Linear (Toeplitz / Yule–Walker) and information (KSG) bounds. **No confidence interval** — block
bootstrapping a seasonal series requires phase- and amplitude-preserving resampling we did not
implement, and the naive version is biased low in proportion to seasonal share. The point
estimator is validated against closed-form AR(2) solutions (error < 0.003; see Cell 4v).

In [ ]:
from scipy.linalg import toeplitz, solve_toeplitz
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors

mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
diag = pd.read_parquet(OUT/"stageA_diagnostics.parquet")
LAGS_LIN, LAGS_MI = 12, [1,2,12]

def deseason(y, n_h=2):
    t = np.arange(len(y))
    H = np.column_stack([np.ones(len(y))] +
        [f(2*np.pi*k*t/12) for k in range(1,n_h+1) for f in (np.sin,np.cos)])
    b, *_ = np.linalg.lstsq(H, y, rcond=None); return y - H@b

def ceiling_linear(y, p=LAGS_LIN):
    gam = acf(y, nlags=p, fft=True) * y.var()
    try: w = solve_toeplitz((gam[:p], gam[:p]), gam[1:p+1])
    except Exception: w = np.linalg.lstsq(toeplitz(gam[:p]), gam[1:p+1], rcond=None)[0]
    return float(np.clip(1 - (gam[0] - w@gam[1:p+1])/gam[0], 0, 1))

# KSG with tie-breaking (paper Sec 2.4), as run. Hyper-arid cells sit at the 0.04 floor for
# most months; exact ties collapse the k-NN radii and the plain estimator returns inf.
# Fix: 1e-8 s.d. Gaussian jitter on the standardised variables, MI capped at 5 nats.
# ONE generator, seeded 0, consumed raw-then-deseasonalised for each cell in groupby order:
# keep that order or the jitter draws (and the hyper-arid MI values) will not reproduce.
rng_j = np.random.default_rng(0)
def ksg_mi(X, y, k=4, jitter=1e-8):
    X = (X-X.mean(0))/(X.std(0)+1e-12); y = (y-y.mean())/(y.std()+1e-12)
    X = X + rng_j.normal(0, jitter, X.shape)      # break exact ties
    y = y + rng_j.normal(0, jitter, y.shape)
    Z = np.column_stack([X, y]); n = len(Z)
    eps = NearestNeighbors(n_neighbors=k+1, metric="chebyshev").fit(Z).kneighbors(Z)[0][:,k]
    eps = np.maximum(eps, 1e-12)
    def cnt(A):
        nn = NearestNeighbors(metric="chebyshev").fit(A)
        return np.array([len(nn.radius_neighbors(A[j:j+1], radius=eps[j]*(1-1e-9),
                         return_distance=False)[0])-1 for j in range(n)])
    nx, ny = np.maximum(cnt(X),0), np.maximum(cnt(y[:,None]),0)
    mi = digamma(k) + digamma(n) - np.mean(digamma(nx+1) + digamma(ny+1))
    return float(np.clip(mi, 0.0, 5.0))           # 5 nats -> R2 bound 0.99995

def lagmat(y, lags):
    m = max(lags); return np.column_stack([y[m-L:len(y)-L] for L in lags]), y[m:]

rows = []
for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
    y = gg.sort_values("month").GWETTOP.to_numpy(float); yd = deseason(y)
    Xr, Yr = lagmat(y, LAGS_MI); Xd, Yd = lagmat(yd, LAGS_MI)
    mi_r, mi_d = ksg_mi(Xr, Yr), ksg_mi(Xd, Yd)
    rows.append(dict(cell_id=cid, ceil_lin_raw=ceiling_linear(y), ceil_lin_des=ceiling_linear(yd),
        mi_raw=mi_r, mi_des=mi_d,
        ceil_mi_raw=float(np.clip(1-np.exp(-2*mi_r),0,1)),
        ceil_mi_des=float(np.clip(1-np.exp(-2*mi_d),0,1))))
    if ci % 50 == 0: print(f"  cell {ci}")

ceil = pd.DataFrame(rows).merge(diag[["cell_id","name","AI","aridity_class","irrig_index",
    "irrig_class","hemi","seas_share","acf1_des","sampen_des","replication"]], on="cell_id")
ceil.to_parquet(OUT/"stageB_ceilings.parquet", index=False)

print("\n--- SANITY ---")
for c in ["ceil_lin_raw","ceil_lin_des","ceil_mi_raw","ceil_mi_des"]:
    print(f"{c}: {ceil[c].min():.3f}-{ceil[c].max():.3f} in [0,1]? {bool(ceil[c].between(0,1).all())}")
print("MI finite everywhere?", bool(np.isfinite(ceil[["mi_raw","mi_des"]]).all().all()))
print("raw >= des everywhere?", bool((ceil.ceil_lin_raw >= ceil.ceil_lin_des-1e-9).all()))
print("panel mean ceiling: %.3f" % ceil.ceil_lin_raw.mean())
print("\nby aridity:\n", ceil.groupby("aridity_class",observed=True)[["ceil_lin_raw","ceil_lin_des","seas_share"]].mean().round(3).to_string())
print("\ncorr(ceil_lin_des, acf1_des)   = %.3f" % ceil[["ceil_lin_des","acf1_des"]].corr().iloc[0,1])
print("corr(ceil_lin_raw, seas_share) = %.3f" % ceil[["ceil_lin_raw","seas_share"]].corr().iloc[0,1])
print("\nPunjab:\n", ceil[ceil.replication][["name","ceil_lin_raw","ceil_lin_des","ceil_mi_raw"]].round(3).to_string(index=False))

## Cell 4v — validate the ceiling estimator against analytic AR(2)

In [ ]:
rng_v = np.random.default_rng(0)
print("AR(2) validation — true R^2 from closed form vs estimator:")
for (a1,a2,sig) in [(0.6,0.2,1.0),(0.9,-0.3,1.0),(0.3,0.0,1.0)]:
    n=100000; e=rng_v.normal(0,sig,n); y=np.zeros(n)
    for t in range(2,n): y[t]=a1*y[t-1]+a2*y[t-2]+e[t]
    y=y[1000:]; true_r2=1-sig**2/y.var(); est=ceiling_linear(y)
    print(f"  AR2({a1},{a2}): true {true_r2:.4f} | est {est:.4f} | diff {est-true_r2:+.4f}")
print("\n-> all |diff| should be < 0.005")

## Cell 5 — Stage C: benchmark ladder (checkpointed)

`SEEDS = 5` (as reported). 249 cells × 5 folds × 6 learned models × 5 seeds, plus 3 floors. Budget 40–90 min. **Run Cell Z after this.**

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
import warnings; warnings.filterwarnings("ignore")
try:
    from xgboost import XGBRegressor; HAS_XGB = True
except ImportError: HAS_XGB = False
try:
    from lightgbm import LGBMRegressor; HAS_LGB = True
except ImportError: HAS_LGB = False

CK_C = OUT/"stageC_ck"; CK_C.mkdir(exist_ok=True)
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
LAGS = list(range(1,13)); N_FOLDS, TEST_LEN, SEEDS = 5, 60, 5

def nse(o,p): return 1 - ((o-p)**2).sum()/((o-o.mean())**2).sum()
def kge(o,p):
    r = np.corrcoef(o,p)[0,1]
    return 1 - np.sqrt((r-1)**2 + (p.std()/o.std()-1)**2 + (p.mean()/o.mean()-1)**2)
def design(y, months):
    m = max(LAGS)
    return np.column_stack([y[m-L:len(y)-L] for L in LAGS]), y[m:], months[m:]

def make_models(seed):
    M = {"AR12": LinearRegression(),
         "CART": DecisionTreeRegressor(random_state=seed),
         "RF": RandomForestRegressor(n_estimators=300, min_samples_leaf=4, random_state=seed, n_jobs=-1),
         "AdaBoost-CART": AdaBoostRegressor(DecisionTreeRegressor(max_depth=6, random_state=seed),
                                            n_estimators=100, random_state=seed)}
    if HAS_XGB: M["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                            subsample=0.8, random_state=seed, verbosity=0)
    if HAS_LGB: M["LightGBM"] = LGBMRegressor(n_estimators=300, num_leaves=31,
                                              learning_rate=0.05, random_state=seed, verbose=-1)
    return M

for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
    fpk = CK_C/f"{cid}.parquet"
    if fpk.exists(): continue
    gg = gg.sort_values("month")
    X, Y, Mo = design(gg.GWETTOP.to_numpy(float), gg.month.to_numpy()); n = len(Y)
    rr = []
    for f in range(N_FOLDS):
        te_hi = n-(N_FOLDS-1-f)*TEST_LEN; te_lo = te_hi-TEST_LEN
        if te_lo < 120: continue
        Xtr,Ytr,Xte,Yte = X[:te_lo],Y[:te_lo],X[te_lo:te_hi],Y[te_lo:te_hi]
        mo_tr = pd.DatetimeIndex(Mo[:te_lo]).month; mo_te = pd.DatetimeIndex(Mo[te_lo:te_hi]).month
        clim = pd.Series(Ytr).groupby(mo_tr).mean()
        for nm,p in {"Climatology":clim.reindex(mo_te).to_numpy(),
                     "Persistence":Xte[:,0], "SeasonalNaive":Xte[:,11]}.items():
            rr.append(dict(cell_id=cid,fold=f,model=nm,seed=0,nse=nse(Yte,p),kge=kge(Yte,p)))
        for s in range(SEEDS):
            for nm,mdl in make_models(s).items():
                if nm=="AR12" and s>0: continue
                mdl.fit(Xtr,Ytr); p=mdl.predict(Xte)
                rr.append(dict(cell_id=cid,fold=f,model=nm,seed=s,nse=nse(Yte,p),kge=kge(Yte,p)))
    pd.DataFrame(rr).to_parquet(fpk, index=False)
    if ci % 25 == 0: print(f"  cell {ci}/{mon.cell_id.nunique()}")

res = pd.concat([pd.read_parquet(f) for f in sorted(CK_C.glob("*.parquet"))], ignore_index=True)
res.to_parquet(OUT/"stageC_legacy_results.parquet", index=False)

print("\n--- SANITY ---")
print("cells:", res.cell_id.nunique(), "| rows:", len(res), "| non-finite:", int((~np.isfinite(res.nse)).sum()))
print("\nmean NSE by model:\n", res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
cb = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top = cb.loc[cb.groupby("cell_id").nse.idxmax()]
print("\nwins by cell:\n", top.model.value_counts().to_string())
ml = res[~res.model.isin(["Climatology","Persistence","SeasonalNaive","AR12"])]
seed_sd = ml.groupby(["cell_id","fold","model"]).nse.std().dropna().median()
betw_sd = ml.groupby(["cell_id","fold"]).apply(lambda z: z.groupby("model").nse.mean().std()).dropna().median()
print("\n*** seed sd %.4f | between-model sd %.4f | ratio %.1fx" % (seed_sd, betw_sd, betw_sd/seed_sd))
t = top.merge(ceil, on="cell_id"); t["gap"] = t.ceil_lin_raw - t.nse
t.to_parquet(OUT/"stageC_vs_ceiling.parquet", index=False)
print("*** corr(best NSE, ceiling) = %.3f | mean gap %.3f | above ceiling %d/%d"
      % (t[["nse","ceil_lin_raw"]].corr().iloc[0,1], t.gap.mean(), int((t.gap<0).sum()), len(t)))

## Cell 6 — tuning-free tier: ESN (+ optional Chronos zero-shot)

The ESN readout is 301-dimensional; inputs and target are standardised and the ridge penalty is
selected on an inner holdout. A weak fixed ridge diverges — do not remove this.

Set `RUN_CHRONOS = True` on a GPU runtime for the zero-shot foundation-model comparison.

In [ ]:
RUN_CHRONOS = False    # True after: !pip install -q chronos-forecasting  (use a GPU runtime)
ESN_SEEDS = 5
CK_E = OUT/"stageE_ck"; CK_E.mkdir(exist_ok=True)

class ESN:
    def __init__(self, n_res=300, rho=0.9, sparsity=0.1, leak=0.3,
                 ridge=None, in_scale=0.5, seed=0):
        self.p=dict(n_res=n_res,rho=rho,leak=leak,in_scale=in_scale)
        self.ridge=ridge; self.sparsity=sparsity; self.seed=seed
    def _init(self,n_in):
        N=self.p["n_res"]; r=np.random.default_rng(self.seed)
        W=r.normal(0,1,(N,N))*(r.random((N,N))<self.sparsity)
        ev=np.max(np.abs(np.linalg.eigvals(W)))
        self.W=W*(self.p["rho"]/ev if ev>0 else 1.0)
        self.Win=r.uniform(-0.5,0.5,(N,n_in))*self.p["in_scale"]
    def _states(self,X):
        N,a=self.p["n_res"],self.p["leak"]; s=np.zeros(N); S=np.empty((len(X),N))
        for i,x in enumerate(X):
            s=(1-a)*s+a*np.tanh(self.Win@x+self.W@s); S[i]=s
        return S
    def _readout(self,S,y,lam):
        A=np.column_stack([S,np.ones(len(S))])
        return np.linalg.solve(A.T@A+lam*np.eye(A.shape[1]),A.T@y)
    def fit(self,X,y):
        self.mu,self.sd=X.mean(0),X.std(0)+1e-9; self.ym,self.ys=y.mean(),y.std()+1e-9
        Xs,ys_=(X-self.mu)/self.sd,(y-self.ym)/self.ys
        self._init(Xs.shape[1]); S=self._states(Xs)
        if self.ridge is None:
            cut=int(len(S)*0.8); best,blam=-np.inf,1.0
            for lam in [1e-2,1e-1,1.0,10.0,100.0]:
                w=self._readout(S[:cut],ys_[:cut],lam)
                p=np.column_stack([S[cut:],np.ones(len(S)-cut)])@w
                sc=-np.mean((p-ys_[cut:])**2)
                if sc>best: best,blam=sc,lam
            self.lam_=blam
        else: self.lam_=self.ridge
        self.Wout=self._readout(S,ys_,self.lam_); return self
    def predict(self,X):
        S=self._states((X-self.mu)/self.sd)
        return (np.column_stack([S,np.ones(len(S))])@self.Wout)*self.ys+self.ym

pipe=None
if RUN_CHRONOS:
    import torch
    from chronos import BaseChronosPipeline
    dev="cuda" if torch.cuda.is_available() else "cpu"
    pipe=BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base",
                                             device_map=dev,torch_dtype=torch.float32)
    print("chronos on",dev)

for ci,(cid,gg) in enumerate(mon.groupby("cell_id")):
    fpk=CK_E/f"{cid}.parquet"
    if fpk.exists(): continue
    gg=gg.sort_values("month"); y=gg.GWETTOP.to_numpy(float)
    X,Y,_=design(y,gg.month.to_numpy()); m0=max(LAGS); n=len(Y); rr=[]
    for f in range(N_FOLDS):
        te_hi=n-(N_FOLDS-1-f)*TEST_LEN; te_lo=te_hi-TEST_LEN
        if te_lo<120: continue
        Xtr,Ytr,Xte,Yte=X[:te_lo],Y[:te_lo],X[te_lo:te_hi],Y[te_lo:te_hi]
        for s in range(ESN_SEEDS):
            p=ESN(seed=s).fit(Xtr,Ytr).predict(Xte)
            rr.append(dict(cell_id=cid,fold=f,model="ESN",seed=s,nse=nse(Yte,p),kge=kge(Yte,p)))
        if pipe is not None:
            ctx=[torch.tensor(y[:m0+te_lo+t],dtype=torch.float32) for t in range(len(Yte))]
            q,_m=pipe.predict_quantiles(context=ctx,prediction_length=1,quantile_levels=[0.5])
            p=q[:,0,0].numpy()
            rr.append(dict(cell_id=cid,fold=f,model="Chronos-Bolt(0shot)",seed=0,
                           nse=nse(Yte,p),kge=kge(Yte,p)))
    pd.DataFrame(rr).to_parquet(fpk,index=False)
    if ci%25==0: print(f"  cell {ci}")

new=pd.concat([pd.read_parquet(f) for f in sorted(CK_E.glob("*.parquet"))],ignore_index=True)
res=pd.concat([pd.read_parquet(OUT/"stageC_legacy_results.parquet"),new],ignore_index=True)
res.to_parquet(OUT/"stageC_all_results.parquet",index=False)

print("\n--- SANITY ---")
esn=res[res.model=="ESN"]
print("ESN mean %.3f | min %.3f | frac<0 %.2f" % (esn.nse.mean(),esn.nse.min(),(esn.nse<0).mean()))
print("  -> if mean is large-negative or min < -5, the reservoir diverged; DO NOT report")
print("\nmean NSE:\n", res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
tuned=res[res.model.isin(["RF","XGBoost","LightGBM"])].nse.mean()
free =res[res.model.isin(["ESN","Chronos-Bolt(0shot)"])].nse.mean()
print("\ntuned tree tier %.3f | tuning-free tier %.3f | diff %+.3f" % (tuned,free,tuned-free))
print("AR12 %.3f | climatology floor %.3f"
      % (res[res.model=="AR12"].nse.mean(), res[res.model=="Climatology"].nse.mean()))

## Cell 7 — Stage D (PRIMARY): drivers at lags 1–3

Forecast-realistic. This is the headline driver result.

In [ ]:
DRV = ["PRECTOTCORR","T2M","T2M_MAX","RH2M","WS2M","ALLSKY_SFC_SW_DWN"]

def run_stageD(drv_lags, ck_name, out_name):
    CK = OUT/ck_name; CK.mkdir(exist_ok=True)
    def build(gg):
        y=gg.GWETTOP.to_numpy(float); m=max(max(LAGS),max(drv_lags))
        Xlag=np.column_stack([y[m-L:len(y)-L] for L in LAGS])
        cols=[]
        for d in DRV:
            v=gg[d].to_numpy(float)
            for L in drv_lags: cols.append(v[m-L:len(v)-L])
        return Xlag,np.column_stack(cols),y[m:]
    def dmodels(seed):
        M={"AR/OLS":LinearRegression(),
           "RF":RandomForestRegressor(n_estimators=300,min_samples_leaf=4,random_state=seed,n_jobs=-1)}
        if HAS_LGB: M["LightGBM"]=LGBMRegressor(n_estimators=300,num_leaves=31,
                                                learning_rate=0.05,random_state=seed,verbose=-1)
        return M
    for ci,(cid,gg) in enumerate(mon.groupby("cell_id")):
        fpk=CK/f"{cid}.parquet"
        if fpk.exists(): continue
        gg=gg.sort_values("month"); Xlag,Xdrv,Y=build(gg); n=len(Y)
        SETS={"lag_only":Xlag,"lag+drivers":np.column_stack([Xlag,Xdrv]),"drivers_only":Xdrv}
        rr=[]
        for f in range(N_FOLDS):
            te_hi=n-(N_FOLDS-1-f)*TEST_LEN; te_lo=te_hi-TEST_LEN
            if te_lo<120: continue
            for sn,XX in SETS.items():
                for s in range(SEEDS):
                    for mn,mdl in dmodels(s).items():
                        if mn=="AR/OLS" and s>0: continue
                        mdl.fit(XX[:te_lo],Y[:te_lo]); p=mdl.predict(XX[te_lo:te_hi])
                        rr.append(dict(cell_id=cid,fold=f,inputs=sn,model=mn,seed=s,
                                       nse=nse(Y[te_lo:te_hi],p)))
        pd.DataFrame(rr).to_parquet(fpk,index=False)
        if ci%25==0: print(f"  cell {ci}")
    d=pd.concat([pd.read_parquet(f) for f in sorted(CK.glob("*.parquet"))],ignore_index=True)
    best=d.groupby(["cell_id","inputs","model"]).nse.mean().reset_index()
    best=best.loc[best.groupby(["cell_id","inputs"]).nse.idxmax()]
    piv=best.pivot(index="cell_id",columns="inputs",values="nse").join(
        ceil.set_index("cell_id")[["name","ceil_lin_raw","acf1_des","sampen_des","seas_share",
                                   "irrig_class","aridity_class","hemi","replication"]])
    piv["driver_gain"]=piv["lag+drivers"]-piv["lag_only"]
    piv["above_ceiling"]=piv["lag+drivers"]-piv["ceil_lin_raw"]
    piv.to_parquet(OUT/out_name)
    return d,piv

d1,sd1 = run_stageD([1,2,3], "stageD_ck_lag1", "stageD_summary_lag1.parquet")

print("\n--- SANITY (FORECAST-HONEST, lags 1-3) ---")
print("cells:", d1.cell_id.nunique())
print("mean NSE by input set:\n", d1.groupby("inputs").nse.mean().round(3).to_string())
print("\nmean driver gain: %.3f" % sd1.driver_gain.mean())
print("Wilcoxon: p = %.2g" % stats.wilcoxon(sd1["lag+drivers"], sd1["lag_only"])[1])
print("cells above lag-only ceiling: %d / %d" % (int((sd1.above_ceiling>0).sum()), len(sd1)))
for v in ["acf1_des","sampen_des"]:
    r_,p_ = stats.pearsonr(sd1[v].astype(float), sd1.driver_gain.astype(float))
    print("corr(%-11s, driver_gain) = %+.3f (p=%.2g)" % (v,r_,p_))
print("\nby aridity:\n", sd1.groupby("aridity_class",observed=True)[["lag_only","lag+drivers","driver_gain"]].mean().round(3).to_string())
print("\nPunjab:\n", sd1[sd1.replication][["name","lag_only","lag+drivers","driver_gain","ceil_lin_raw","above_ceiling"]].round(3).to_string())

## Cell 7b — Stage D (DIAGNOSTIC): drivers at lags 0–2

Includes the concurrent month. Reported as an information-content diagnostic **only** — this
configuration recovers SSW from the meteorology that generates it inside the same land-surface
model, and is not a forecast.

In [ ]:
d0,sd0 = run_stageD([0,1,2], "stageD_ck_lag0", "stageD_summary_lag0.parquet")

print("\n--- SANITY (DIAGNOSTIC, lags 0-2) ---")
print("mean NSE by input set:\n", d0.groupby("inputs").nse.mean().round(3).to_string())
print("\n*** drivers_only here should be MUCH higher than in Cell 7 — that gap is the leakage")
print("mean driver gain: %.3f  (honest variant gave %.3f)" % (sd0.driver_gain.mean(), sd1.driver_gain.mean()))
print("cells above ceiling: %d/%d  (honest gave %d/%d)"
      % (int((sd0.above_ceiling>0).sum()), len(sd0), int((sd1.above_ceiling>0).sum()), len(sd1)))
r0,p0 = stats.pearsonr(sd0.acf1_des.astype(float), sd0.driver_gain.astype(float))
r1,p1 = stats.pearsonr(sd1.acf1_des.astype(float), sd1.driver_gain.astype(float))
print("\ncorr(acf1_des, driver_gain): leaked %+.3f (p=%.2g) | honest %+.3f (p=%.2g)" % (r0,p0,r1,p1))
print("  -> if the leaked version is significant and the honest one is not, that IS the finding")

## Cell 8 — ceiling-exceedance diagnostic

In [ ]:
res  = pd.read_parquet(OUT/"stageC_all_results.parquet")
best = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top  = best.loc[best.groupby("cell_id").nse.idxmax()].merge(ceil, on="cell_id")
top["exceed"] = top.nse - top.ceil_lin_raw
top["over"]   = top.exceed > 0

print("cells above linear ceiling: %d / %d" % (int(top.over.sum()), len(top)))
print("exceedance %+.3f to %+.3f | mean over-cells %+.3f"
      % (top.exceed.min(), top.exceed.max(), top.loc[top.over,"exceed"].mean() if top.over.any() else np.nan))
print("\nover vs not:\n", top.groupby("over")[["seas_share","acf1_des","sampen_des","ceil_lin_raw","AI"]].mean().round(3).to_string())
for v in ["seas_share","acf1_des","sampen_des","AI"]:
    r_,p_ = stats.pearsonr(top[v].astype(float), top.exceed.astype(float))
    print("  corr(%-11s, exceedance) = %+.3f (p=%.3g)" % (v,r_,p_))
print("\nwhich model exceeds (nonlinear should dominate if genuine):\n",
      pd.crosstab(top.model, top.over).to_string())
top.to_parquet(OUT/"exceedance_diagnostic.parquet", index=False)

## Cell 9 — figures (300 dpi, as submitted)

The final figures are rendered by `scripts/make_figures.py`. Run the notebook from the repository root, or upload the `scripts/` folder to Colab.

In [ ]:
import sys, numpy as np
sys.path.insert(0, ".") if Path("scripts").exists() else sys.path.insert(0, "..")
from scripts.make_figures import main as make_figures
make_figures(OUT, Path("figures"))

# quantities Cell 10 prints
ml = res[~res.model.isin(["Climatology","Persistence","SeasonalNaive","AR12"])]
seed_sd = ml.groupby(["cell_id","fold","model"]).nse.std().dropna()
betw_sd = (ml.groupby(["cell_id","fold","model"]).nse.mean()
             .groupby(level=["cell_id","fold"]).std().dropna())
rr_ = np.corrcoef(top.ceil_lin_raw, top.nse)[0,1]
pan = pd.read_parquet(OUT/"panel_global.parquet")
sd1 = pd.read_parquet(OUT/"stageD_summary_lag1.parquet")
sd0 = pd.read_parquet(OUT/"stageD_summary_lag0.parquet")

## Cell 10 — headline numbers (paste this output back)

In [ ]:
print("="*66); print("GLOBAL HEADLINE NUMBERS"); print("="*66)
pan = pan[pan.cell_id.isin(res.cell_id.unique())]   # analysed cells only (degenerate cell excluded)
print("\npanel N = %d cells | %d models | %d strata | hemi %s" % (res.cell_id.nunique(), res.model.nunique(), pan.stratum.nunique(), pan.hemi.value_counts().to_dict()))
print("lat %.1f to %.1f | AI %.3f-%.3f | P %.0f-%.0f mm/yr"
      % (pan.lat.min(),pan.lat.max(),pan.AI.min(),pan.AI.max(),pan.P_ann.min(),pan.P_ann.max()))
print("corr(AI, irrig_index) = %.3f" % stats.pearsonr(pan.AI,pan.irrig_index)[0])

print("\n[1] CEILING")
print("  panel mean %.3f | range %.3f-%.3f" % (ceil.ceil_lin_raw.mean(),ceil.ceil_lin_raw.min(),ceil.ceil_lin_raw.max()))
print("  corr(ceil_des, acf1_des) = %.3f" % ceil[["ceil_lin_des","acf1_des"]].corr().iloc[0,1])
print(ceil[ceil.replication][["name","ceil_lin_raw","ceil_lin_des","seas_share"]].round(3).to_string(index=False))

print("\n[2] BENCHMARK LADDER")
print(res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
print("\nwins by cell:\n", top.model.value_counts().to_string())

print("\n[3] SEED vs MODEL")
print("  seed sd %.4f | between-model sd %.4f | ratio %.1fx" % (seed_sd.median(),betw_sd.median(),betw_sd.median()/seed_sd.median()))
print("  corr(best NSE, ceiling) = %.3f" % rr_)

print("\n[4] STAGE D")
for tag,s_ in [("lag 1-3 (FORECAST)",sd1),("lag 0-2 (DIAGNOSTIC)",sd0)]:
    r_,p_ = stats.pearsonr(s_.acf1_des.astype(float), s_.driver_gain.astype(float))
    print("  %-22s %.3f -> %.3f | gain %+.3f | above ceiling %d/%d | corr(acf1,gain) %+.3f (p=%.2g)"
          % (tag, s_["lag_only"].mean(), s_["lag+drivers"].mean(), s_.driver_gain.mean(),
             int((s_.above_ceiling>0).sum()), len(s_), r_, p_))
print("  NOTE: lag_only / lag+drivers above = BEST model per cell, averaged over cells.")
print("  Mean over all models x seeds x folds (a different basis):")
for tag,d_ in [("lag 1-3",d1),("lag 0-2",d0)]:
    print("   ", tag, d_.groupby("inputs").nse.mean().round(3).to_dict())
print("\n  Punjab, forecast-honest:")
print(sd1[sd1.replication][["name","lag_only","lag+drivers","driver_gain","ceil_lin_raw","above_ceiling"]].round(3).to_string(index=False))

print("\n[5] EXCEEDANCE")
print("  %d/%d cells above ceiling | corr(seas_share, exceedance) = %.3f"
      % (int(top.over.sum()), len(top), stats.pearsonr(top.seas_share.astype(float), top.exceed.astype(float))[0]))